# PaMCP Core — Master Colab (resume-first)

**Single entry point for the PaMCP experimental programme.** It inventories committed GitHub outputs first, validates cache provenance where possible, reuses compatible results, and schedules only missing/invalid stages. Historical exploratory outputs are preserved for inspection but are never silently promoted to confirmatory evidence.

**Important:** a cached file is not automatically a valid result. The notebook distinguishes `REUSE_VALID`, `REUSE_HISTORICAL_ONLY`, `RUN_REQUIRED`, and `BLOCKED`. It does not replay a task with a valid result. It does not invent missing oracle labels or claim that adapter tests are scientific results.

Protocol: [PAMCP_CORE_PROTOCOL_V2.md](https://github.com/msellamiTN/taas-provenance-discovery/blob/research/pamcp-multibenchmark-v1/docs/experiment_design/PAMCP_CORE_PROTOCOL_V2.md).

In [ ]:
# Robust, restart-safe checkout. Avoid checkout -B FETCH_HEAD.
import os, sys, json, subprocess, hashlib, pathlib, shutil, zipfile, time, platform
from datetime import datetime, timezone
REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
BRANCH = "research/pamcp-multibenchmark-v1"
ROOT = pathlib.Path("/content/taas-provenance-discovery")
if ROOT.exists() and (ROOT / ".git").exists():
    subprocess.run(["git","-C",str(ROOT),"fetch","--depth","1","origin",f"refs/heads/{BRANCH}:refs/remotes/origin/pamcp-v2"],check=True)
    subprocess.run(["git","-C",str(ROOT),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(ROOT),"reset","--hard","refs/remotes/origin/pamcp-v2"],check=True)
else:
    if ROOT.exists(): raise RuntimeError(f"{ROOT} exists but is not a Git checkout")
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Commit:",COMMIT)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[experiment,benchmark-hf,test]"],check=True)


In [ ]:
# Validate implementation contracts before running experiments.
TESTS=[
 "tests/test_pamcp_benchmark_io.py",
 "tests/test_pamcp_benchmark_adapters.py",
 "tests/test_mcp_atlas_adapter.py",
 "tests/test_toolret_adapter.py",
 "tests/test_evaluation_metrics.py",
 "tests/test_declared_effects_are_not_observations.py"
]
existing=[p for p in TESTS if (ROOT/p).exists()]
subprocess.run([sys.executable,"-m","pytest","-q",*existing],check=True)
print("Validated tests:",existing)


## 1. Immutable run manifest and cached-result policy

A prior ToolRet run is reused only when its source revisions, query/corpus fingerprints, adapter/metric contract and parameters are compatible. Otherwise the notebook reports a cache miss; it does not silently relabel an old run.

In [ ]:
OUT=ROOT/"results"/"pamcp_core_v2"
OUT.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+COMMIT[:8]
RUN=OUT/RUN_ID
RUN.mkdir(parents=True,exist_ok=False)
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
run_manifest={
 "protocol":"PaMCP Core Experimental Protocol v2.0",
 "run_id":RUN_ID,"commit":COMMIT,"branch":BRANCH,
 "timestamp_utc":datetime.now(timezone.utc).isoformat(),
 "python":platform.python_version(),"platform":platform.platform(),
 "cache_policy":"reuse only with exact provenance/parameter/metric compatibility",
 "results":{},"blocked":{}
}
(RUN/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2))
print("Run directory:",RUN)


In [ ]:
# Resume-first cache inventory: inspect committed outputs before any expensive stage.
import hashlib
FORCE_RERUN = os.environ.get('PAMCP_FORCE_RERUN', '0').lower() in {'1','true','yes'}
HISTORICAL_CACHE = {
  'E0_registry_exploratory': ['results/e0_registry.json'],
  'E2_eligibility_exploratory': ['results/e2_eligibility.json'],
  'E3_gap_exploratory': ['results/e3_gap_sonnet.json', 'results/e3_tune.json'],
  'E4_injection_exploratory': ['results/e4_injection.json'],
  'E5_synthetic_exploratory': ['results/e5_synthetic.json'],
  'E6_ablations_exploratory': ['results/e6_ablations.json'],
  'E7_scalability_exploratory': ['results/e7_scalability.json'],
  'ToolRet_lexical_baselines': ['artifacts/multibenchmark/toolret_baselines/aggregate_metrics.json'],
}
def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def cache_inventory():
    inventory={}
    for stage, relpaths in HISTORICAL_CACHE.items():
        found=[]
        for rel in relpaths:
            p=ROOT/rel
            if p.is_file(): found.append({'path':rel,'sha256':file_sha256(p),'bytes':p.stat().st_size})
        if found:
            inventory[stage]={'decision':'REUSE_HISTORICAL_ONLY','files':found,
              'reason':'Committed outputs exist, but this cache map does not establish compatibility with the frozen v2 confirmatory protocol.'}
        else:
            inventory[stage]={'decision':'RUN_REQUIRED','files':[],'reason':'No committed output found.'}
    return inventory
CACHE=cache_inventory()
run_manifest['github_cache_inventory']=CACHE
run_manifest['force_rerun']=FORCE_RERUN
for stage, entry in CACHE.items(): print(f"{stage}: {entry['decision']} ({len(entry['files'])} file(s))")
print('FORCE_RERUN =', FORCE_RERUN)

# Guard used by subsequent experiment cells. Never overwrite a valid/recorded output by default.
def should_execute(stage, cache_key=None):
    if FORCE_RERUN: return True
    entry=CACHE.get(stage)
    return entry is None or entry['decision']=='RUN_REQUIRED'


## 2. Dataset acquisition and normalization

The first acquisition is the public MCP-Atlas parquet release. The adapter keeps task-specific ENABLED_TOOLS in discovery and GTFA/trajectory references in evaluator-only files. Other dataset adapters are activated only after their exact source revisions, licenses and schemas are validated.

In [ ]:
from pathlib import Path
from taas.experiments.mcp_atlas_adapter import adapt_mcp_atlas_parquet
RAW=ROOT/"data"/"raw"/"mcp_atlas_public"
RAW.mkdir(parents=True,exist_ok=True)
# Hugging Face CLI download; pin a dataset revision in the run manifest.
HF_REVISION=os.environ.get("MCP_ATLAS_REVISION","")
if not HF_REVISION:
    print("MCP_ATLAS_REVISION not set: using current dataset snapshot is NOT publication-grade; set an immutable HF commit before confirmatory execution.")
else:
    subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub"],check=True)
    subprocess.run(["hf","download","ScaleAI/MCP-Atlas","--repo-type","dataset","--revision",HF_REVISION,"--local-dir",str(RAW)],check=True)
    parquet=sorted(RAW.glob("*.parquet"))
    if len(parquet)!=1: raise RuntimeError(f"Expected one parquet file, found {len(parquet)}")
    atlas_out=ROOT/"data"/"processed"/"mcp_atlas_public_v2"
    atlas_manifest=adapt_mcp_atlas_parquet(parquet[0],atlas_out)
    run_manifest["results"]["mcp_atlas_public"]={"status":"ADAPTED","revision":HF_REVISION,"manifest":atlas_manifest}
    print(atlas_manifest["counts"])


## 3. Reuse completed ToolRet baselines

The repository contains previously committed BM25/TF-IDF/Random aggregate outputs and plots. This cell records them as historical completed outputs; it does not rerun or claim new execution. Per-task rankings and source manifests are required before using these outputs for paired tests or new comparisons.

In [ ]:
import json
toolret_dir=ROOT/"artifacts"/"multibenchmark"/"toolret_baselines"
agg=toolret_dir/"aggregate_metrics.json"
if agg.exists():
    cached=json.loads(agg.read_text())
    run_manifest["results"]["toolret_existing"]={
      "status":"EXISTING_RECORDED_OUTPUT",
      "aggregate_sha256":sha256(agg),
      "aggregate_path":str(agg),
      "note":"Reuse as prior run only; validate detailed task/ranking/qrel fingerprints before paired inference."
    }
    print("Existing ToolRet aggregate found; hash:",sha256(agg))
else:
    run_manifest["blocked"]["toolret_existing"]="No committed aggregate output found"


## 4. E1 registry characterization

Run the existing E1 protocol runner. Its own gate requires independent human labels for inferential claims. Descriptive counts and the scientific completion status are kept distinct.

In [ ]:
e1_script=ROOT/"experiments"/"e1_protocol_runner.py"
if e1_script.exists():
    e1_out=RUN/"e1"
    e1_out.mkdir(exist_ok=True)
    proc=subprocess.run([sys.executable,str(e1_script),"--help"],capture_output=True,text=True)
    print(proc.stdout[:4000] or proc.stderr[:1000])
    print("E1 runner discovered. Execute its documented CLI only after verifying source paths and independent reference-label input.")
    run_manifest["results"]["E1"]={"status":"READY_LOCAL","runner":str(e1_script),"scientific_completion_requires":"independent reference labels"}
else:
    run_manifest["blocked"]["E1"]="E1 runner absent from checked-out commit"


## 5. PaMCP Core factorial variants (G/P/S)

The eight variants are implemented in the repository's variant runner. A preview is not a benchmark score: it must receive normalized task GoalSpecs, contracts and policies, and the evaluator must remain isolated. This cell checks availability and records the required inputs instead of manufacturing task contracts.

In [ ]:
from taas.experiments.catalog import FACTORIAL_VARIANTS
from taas.experiments.variant_runner import run_pamcp_variant
factorial_ids=list(FACTORIAL_VARIANTS)
assert len(factorial_ids)==8, factorial_ids
print("G/P/S variants:",factorial_ids)
run_manifest["results"]["E18"]={
 "status":"READY_LOCAL",
 "variants":factorial_ids,
 "execution_status":"NOT_RUN_NO_FROZEN_INDEPENDENT_TASK_CONTRACT_PACKAGE",
 "required_inputs":["discovery-only GoalSpecs","independently annotated contracts","policy snapshot","evaluator-only oracle"]
}


## 6. Baseline and experiment readiness registry

This is an execution ledger, not a results table. Implementations that are partial or need independent data remain blocked. Do not interpret READY_LOCAL as EXECUTED.

In [ ]:
ledger={
"E1":{"status":"READY_LOCAL","reason":"descriptive registry runner; independent labels needed for inferential labels"},
"E2":{"status":"BLOCKED_ORACLE","reason":"double-annotated semantic typing reference not yet supplied"},
"E3":{"status":"BLOCKED_ORACLE","reason":"independent observed-effect traces and adjudicated labels required"},
"E4-E7":{"status":"BLOCKED_ORACLE","reason":"independent provenance/trust references and attack/evidence labels required"},
"E8":{"status":"EXISTING_RECORDED_OUTPUT","reason":"ToolRet BM25/TF-IDF/Random aggregate exists; exact cache compatibility and per-task files must be validated"},
"E9":{"status":"BLOCKED_DATA","reason":"multi-tool acceptable tool sets required; MCP-Atlas trajectory is diagnostic, not exhaustive gold"},
"E10":{"status":"BLOCKED_ORACLE","reason":"independent composition oracle and contracts required; CMTF shared and independent contract arms"},
"E11":{"status":"BLOCKED_ORACLE","reason":"independent required-capability oracle required"},
"E12":{"status":"BLOCKED_DATA","reason":"frozen calibration corpus and gold outcomes required"},
"E13":{"status":"BLOCKED_EXECUTION","reason":"common runnable sandbox and independent final-state evaluator required"},
"E14":{"status":"PARTIAL_OFFLINE_ONLY","reason":"offline attack datasets can be evaluated; executed attack claims require sandbox"},
"E15":{"status":"READY_LOCAL_WITH_LIMITS","reason":"catalogue scaling possible; synthetic expansion must be reported separately"},
"E16":{"status":"BLOCKED_DATA","reason":"versioned registry snapshots and identity crosswalk required"},
"E17":{"status":"BLOCKED_ORACLE","reason":"independently verified counterexamples and valid compositions required"},
"E18":{"status":"READY_LOCAL","reason":"8 code-level variants exist; independent task/contract package required for confirmatory run"},
"E19":{"status":"BLOCKED_DATA","reason":"common fixed task set and identical model budgets required"},
"E20":{"status":"BLOCKED_DATA","reason":"frozen development set and parameter grid required"},
"E21":{"status":"BLOCKED_DATA","reason":"train/test-disjoint execution trajectories required for AutoTool"},
"E22":{"status":"PENDING","reason":"statistical reproducibility applies after confirmatory runs"}
}
run_manifest["experiment_ledger"]=ledger
print(json.dumps(ledger,indent=2))


## 7. Export reproducibility bundle

This writes the run manifest, readiness ledger and any produced outputs into a ZIP. No model scores are generated by this reporting cell.

In [ ]:
run_manifest["completed_at_utc"]=datetime.now(timezone.utc).isoformat()
(RUN/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2,default=str))
(RUN/"experiment_ledger.json").write_text(json.dumps(ledger,indent=2))
zip_path=ROOT/"results"/f"PaMCP_Core_v2_{RUN_ID}.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(ROOT))
print("Bundle:",zip_path)
print("Commit:",COMMIT)
print("Scientific runs executed in this notebook: only steps explicitly marked executed above; all other rows remain blocked/ready.")
